# Experimento C3: entorno estocástico con distintas probabilidades de destino

### Carga de librerías

In [1]:
import random
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

import entorno
import agentes
import train

### Parámetros del entorno

In [2]:
TAMANOS = {
    'n_estaciones': 4,
    'anclas_por_est': 10,
    'bicis_por_est': 5,    # Estado inicial
    'bicis_por_accion': 2  # Bicis movidas por el agente en c/ accion
}
COSTOS = {
    'accion': 1,
    'accion_invalida': 10,
    'est_vacia': 1,
    'est_llena': 1,
}
TIEMPOS = {
    'viaje': 10,
    'accion': 5,
    'episodio': 8*60,
    'estado': 1,
    'update': 999
}

TRAIN_PARAMS = {
    'alpha_init': 0.1,
    'alpha_step': 1,
    'alpha_end':  0.1,

    'eps_init': 0.7,
    'eps_step': 1,
    'eps_end':  0.7
}

### Dinámica cíclica

In [16]:
PROBS = {
    'origen': 4*[0.1],
    'destino': np.array([
        [0.00, 0.40, 0.30, 0.30],
        [0.05, 0.10, 0.80, 0.05],
        [0.70, 0.10, 0.10, 0.10],
        [0.25, 0.25, 0.25, 0.25]
    ])
}

random.seed(10)

env = entorno.Entorno(TAMANOS, COSTOS, TIEMPOS, PROBS)

agente_Q = agentes.QLearningAgent(action_size=env.action_space.n, alpha=TRAIN_PARAMS['alpha_init'], epsilon=TRAIN_PARAMS['eps_init'])
_, metricas_train_test, counters_train_test, best_qtable = train.train_test(env, agente_Q, params=TRAIN_PARAMS, N=10**5, keep_best=True, verbose=False)

Entrenamiento finalizado en 15.7 minutos.


In [ ]:
# Politica de Q-learning
politica_q = {tuple([int(x) for x in k[:4]]): np.argmax(v) for k,v in agente_Q.q_table.items()}

# Politica de heuristico v2
acciones = [0, (0,1), (0,2), (0,3), (1,0), (1,2), (1,3), (2,0), (2,1), (2,3), (3,0), (3,1), (3,2)]
politica_h2 = {}
for a in range(11):
    for b in range(11):
        for c in range(11):
            for d in range(11):
                if a+b+c+d > 20:
                    continue
                estado = (a,b,c,d)
                if max(estado) >= 7 and min(estado) <= 1:
                    accion = acciones.index((np.argmax(estado), np.argmin(estado)))
                else:
                    accion = 0
                politica_h2[estado] = accion

In [46]:
estados_ambos_agentes = set(politica_q).intersection(politica_h2)
print(f"Nro de estados coincidentes: {len(estados_ambos_agentes)}")

nro_acuerdos = len([s for s in estados_ambos_agentes if politica_q[s] == politica_h2[s] and politica_h2[s] != 0])
print(f"Nro de acuerdos no nulos en esos estados: {nro_acuerdos} ({100*nro_acuerdos/len(estados_ambos_agentes):.1f}%)")

estados_d1 = [s for s in estados_ambos_agentes if s[3] <= 1]
print(f"Nro de estados con D<=1: {len(estados_d1)}")

acuerdos_d1 = [s for s in estados_d1 if politica_q[s] == politica_h2[s] and politica_h2[s] != 0]
print(f"Nro de acuerdos no nulos en esos estados: {len(acuerdos_d1)} ({100*len(acuerdos_d1)/len(estados_d1):.1f}%)")

estados_gt_d1 = [s for s in estados_ambos_agentes if s[3] > 1]
print(f"Nro de estados con D>1: {len(estados_gt_d1)}")

acuerdos_gt_d1 = [s for s in estados_gt_d1 if politica_q[s] == politica_h2[s] and politica_h2[s] != 0]
print(f"Nro de acuerdos no nulos en esos estados: {len(acuerdos_gt_d1)} ({100*len(acuerdos_gt_d1)/len(estados_gt_d1):.1f}%)")

Nro de estados coincidentes: 7631
Nro de acuerdos no nulos en esos estados: 547 (7.2%)
Nro de estados con D<=1: 2065
Nro de acuerdos no nulos en esos estados: 245 (11.9%)
Nro de estados con D>1: 5566
Nro de acuerdos no nulos en esos estados: 302 (5.4%)


In [44]:
print("Q-learning")
dict_q = dict(sorted(dict(Counter([v for k,v in politica_q.items() if k in estados_ambos_agentes])).items()))
print(dict_q)

print("Heuristica v2")
dict_h2 = dict(sorted(dict(Counter([v for k,v in politica_h2.items() if k in estados_ambos_agentes])).items()))
print(dict_h2)

Q-learning
{0: 5872, 1: 203, 2: 54, 3: 200, 4: 84, 5: 49, 6: 155, 7: 206, 8: 285, 9: 375, 10: 57, 11: 69, 12: 22}
Heuristica v2
{0: 3321, 1: 448, 2: 378, 3: 316, 4: 448, 5: 353, 6: 298, 7: 423, 8: 353, 9: 280, 10: 398, 11: 335, 12: 280}


In [6]:
random.seed(777)

env = entorno.Entorno(TAMANOS, COSTOS, TIEMPOS, PROBS)

print("=== Agente Nulo ===")
agente_N = agentes.NullAgent()
_, _, _, _ = train.train(env, agente_N, params=TRAIN_PARAMS, N=10**3)

print("=== Agente Aleatorio ===")
agente_R = agentes.RandomAgent(action_size=env.action_space.n)
_, _, _, _ = train.train(env, agente_R, params=TRAIN_PARAMS, N=10**3)

print("=== Agente Heurístico v1 ===")
agente_H = agentes.HeuristicAgent(action_size=env.action_space.n)
_, _, _, _ = train.train(env, agente_H, params=TRAIN_PARAMS, N=10**3)

print("=== Agente Heurístico v2 ===")
agente_H2 = agentes.HeuristicAgent2(action_size=env.action_space.n)
_, _, counters_h2, _ = train.train(env, agente_H2, params=TRAIN_PARAMS, N=10**3)

print("=== Agente Q-learning ===")
_, metricas_test, counters_test, _ = train.test(env, agente_Q.q_table, N=10**3)

=== Agente Nulo ===
Entrenamiento finalizado en 0.1 minutos.
# Insatisfechos: 25.4 ± 8.7 (15.5 ± 5.8%)
# Prolongados: 13.6 ± 7.9 (8.2 ± 4.8%)
Tiempo desbalanceo: 391.0 ± 101.0 (20.4 ± 5.3%)
Recompensa: -39.0 ± 12.6
=== Agente Aleatorio ===
Entrenamiento finalizado en 0.1 minutos.
# Insatisfechos: 25.9 ± 6.5 (15.7 ± 4.2%)
# Prolongados: 15.0 ± 7.3 (9.1 ± 4.6%)
Tiempo desbalanceo: 357.9 ± 67.0 (18.6 ± 3.5%)
Recompensa: -371.2 ± 74.8
=== Agente Heurístico v1 ===
Entrenamiento finalizado en 0.1 minutos.
# Insatisfechos: 7.3 ± 3.9 (3.9 ± 2.1%)
# Prolongados: 0.0 ± 0.1 (0.0 ± 0.1%)
Tiempo desbalanceo: 85.8 ± 31.2 (4.5 ± 1.6%)
Recompensa: -22.4 ± 5.7
=== Agente Heurístico v2 ===
Entrenamiento finalizado en 0.1 minutos.
# Insatisfechos: 3.9 ± 2.5 (2.0 ± 1.3%)
# Prolongados: 0.2 ± 0.6 (0.1 ± 0.3%)
Tiempo desbalanceo: 53.0 ± 19.4 (2.8 ± 1.0%)
Recompensa: -21.6 ± 4.9
=== Agente Q-learning ===
Entrenamiento finalizado en 0.1 minutos.
# Insatisfechos: 6.8 ± 3.6 (3.7 ± 2.0%)
# Prolongados: 0.3 ± 0.6

### Dinámica absorbente

In [5]:
PROBS = {
    'origen': 4*[0.1],
    'destino': np.array([
        [0.05, 0.10, 0.10, 0.75],
        [0.10, 0.05, 0.10, 0.75],
        [0.10, 0.10, 0.05, 0.75],
        [0.30, 0.30, 0.30, 0.10]
    ])
}

random.seed(10)

env = entorno.Entorno(TAMANOS, COSTOS, TIEMPOS, PROBS)

agente_Q = agentes.QLearningAgent(action_size=env.action_space.n, alpha=TRAIN_PARAMS['alpha_init'], epsilon=TRAIN_PARAMS['eps_init'])
_, metricas_train_test, counters_train_test, best_qtable = train.train_test(env, agente_Q, params=TRAIN_PARAMS, N=10**5, keep_best=True, verbose=False)

Entrenamiento finalizado en 14.5 minutos.


In [8]:
# Politica de Q-learning
politica_q = {tuple([int(x) for x in k[:4]]): np.argmax(v) for k,v in agente_Q.q_table.items()}

print("Q-learning")
dict_q = dict(sorted(dict(Counter(politica_q.values())).items()))
print(f"Total de estados visitados: {len(politica_q)}")
print(dict_q)

print("Heuristica v2")
dict_h2 = dict(sorted(dict(Counter(politica_h2.values())).items()))
print(f"Total de estados visitados: {len(politica_h2)}")
print(dict_h2)

Q-learning
Total de estados visitados: 7623
{0: 5036, 1: 88, 2: 92, 3: 45, 4: 82, 5: 80, 6: 44, 7: 88, 8: 74, 9: 24, 10: 668, 11: 647, 12: 655}
Heuristica v2
Total de estados visitados: 7766
{0: 3456, 1: 448, 2: 378, 3: 316, 4: 448, 5: 353, 6: 298, 7: 423, 8: 353, 9: 280, 10: 398, 11: 335, 12: 280}


In [12]:
random.seed(777)

env = entorno.Entorno(TAMANOS, COSTOS, TIEMPOS, PROBS)

print("=== Agente Nulo ===")
agente_N = agentes.NullAgent()
_, _, _, _ = train.train(env, agente_N, params=TRAIN_PARAMS, N=10**3)

print("=== Agente Aleatorio ===")
agente_R = agentes.RandomAgent(action_size=env.action_space.n)
_, _, _, _ = train.train(env, agente_R, params=TRAIN_PARAMS, N=10**3)

print("=== Agente Heurístico v1 ===")
agente_H = agentes.HeuristicAgent(action_size=env.action_space.n)
_, _, _, _ = train.train(env, agente_H, params=TRAIN_PARAMS, N=10**3)

print("=== Agente Heurístico v2 ===")
agente_H2 = agentes.HeuristicAgent2(action_size=env.action_space.n)
_, _, counters_h2, _ = train.train(env, agente_H2, params=TRAIN_PARAMS, N=10**3)

print("=== Agente Q-learning ===")
_, metricas_test, counters_test, _ = train.test(env, agente_Q.q_table, N=10**3)

=== Agente Nulo ===
Entrenamiento finalizado en 0.1 minutos.
# Insatisfechos: 30.9 ± 8.1 (19.5 ± 5.8%)
# Prolongados: 47.4 ± 13.1 (29.5 ± 8.0%)
Tiempo desbalanceo: 559.9 ± 84.4 (29.2 ± 4.4%)
Recompensa: -78.4 ± 15.2
=== Agente Aleatorio ===
Entrenamiento finalizado en 0.1 minutos.
# Insatisfechos: 32.6 ± 7.6 (20.4 ± 5.3%)
# Prolongados: 29.4 ± 11.7 (18.4 ± 7.6%)
Tiempo desbalanceo: 459.4 ± 75.4 (23.9 ± 3.9%)
Recompensa: -442.6 ± 87.6
=== Agente Heurístico v1 ===
Entrenamiento finalizado en 0.1 minutos.
# Insatisfechos: 6.9 ± 3.6 (3.7 ± 1.9%)
# Prolongados: 0.3 ± 0.7 (0.2 ± 0.4%)
Tiempo desbalanceo: 85.5 ± 29.4 (4.5 ± 1.5%)
Recompensa: -36.2 ± 6.7
=== Agente Heurístico v2 ===
Entrenamiento finalizado en 0.1 minutos.
# Insatisfechos: 4.6 ± 2.6 (2.4 ± 1.4%)
# Prolongados: 3.2 ± 2.4 (1.7 ± 1.3%)
Tiempo desbalanceo: 89.0 ± 26.2 (4.6 ± 1.4%)
Recompensa: -37.0 ± 6.8
=== Agente Q-learning ===
Entrenamiento finalizado en 0.1 minutos.
# Insatisfechos: 8.1 ± 3.9 (4.4 ± 2.2%)
# Prolongados: 0.5 ± 